In [ ]:
pip install mrcfile numpy

In [ ]:
import os
import glob
import numpy as np
import mrcfile
import pandas as pd
from collections import namedtuple
import information_theory as IT

Args = namedtuple("args", ["tomos"])
args = Args("/home/jupyter-vruiz/gdrive_Tomograms")

base_dir = "."
experiment_folders = sorted([f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and f.startswith("epfl_")])
experiment_folders

In [ ]:
results = []

for exp in experiment_folders:
    # Determine the clean name of the experiment (e.g., 'epfl_10nm' from 'epfl_10nm_linear_interpolation')
    # This helps map the denoised tomogram name back to the original file name
    clean_exp_name = "_".join(exp.split("_")[:2]) 
    
    # Define paths to search for denoised folders
    # This handles both 'even_odd' and 'even_odd_registered' workflows
    denoised_search_path = os.path.join(base_dir, exp, "even_odd*", "denoised_vol")
    denoised_folders = glob.glob(denoised_search_path)
    
    for denoised_folder in denoised_folders:
        # Reconstruct the workflow variation (even_odd or even_odd_registered)
        workflow_type = os.path.basename(os.path.dirname(denoised_folder))
        
        # Target denoised file path
        denoised_file = os.path.join(denoised_folder, f"{clean_exp_name}.mrc")
        
        # Original ground truth tomogram is located in the non-interpolated base experiment folder
        original_file = os.path.join(args.tomos, f"{clean_exp_name}.mrc")
        
        # Skip if it is the baseline itself comparing to itself
        #if exp == clean_exp_name:
        #    continue
            
        print(f"Original: {original_file}")
        print(f"Denoised: {denoised_file}")
        with mrcfile.open(original_file, permissive=True) as original, mrcfile.open(denoised_file, permissive=True) as denoised:
        # Load data as float32 to prevent overflow issues during subtraction
            original_data = original.data.astype(np.float32)
            denoised_data = denoised.data.astype(np.float32)
        PSNR = IT.distortion.PSNR(original_data.astype(np.uint8), denoised_data)
        print(f"PSNR={PSNR:.2f}")